# AI工学101 — 第17回

## 特徴量エンジニアリング入門：AIの性能は「何を入力するか」で大きく変わる

前回は、scikit-learn の **Pipeline** を使って、

```
前処理
 ↓
モデル
 ↓
評価
```

という機械学習の一連の流れを一つにまとめました。

今回は、実務でも非常に重要な

> **特徴量エンジニアリング（Feature Engineering）**

を学びます。

実は、多くの機械学習プロジェクトでは、

> **アルゴリズムを変えるより、特徴量を改善したほうが性能が上がる**

ことが珍しくありません。

---

# 🎯 今日のゴール

今回で身につけることは次の5つです。

* 特徴量とは何か説明できる
* 良い特徴量・悪い特徴量を区別できる
* 多項式特徴量の考え方を理解する
* `PolynomialFeatures` を使えるようになる
* 特徴量を増やしすぎる危険性を理解する

---

# 📖 講義（約20分）

## 特徴量（Feature）とは？

例えば住宅価格を予測するとします。

入力候補は

```
広さ

築年数

駅からの距離

部屋数

土地面積
```

などがあります。

これらが

**特徴量（Feature）**

です。

モデルは

```
特徴量

↓

予測価格
```

という流れで学習します。

---

## 良い特徴量とは？

例えば

```
住宅価格
```

を予測するのに

```
部屋数
```

は役立ちそうです。

一方で

```
家の郵便番号の最後の数字
```

は、価格との関係がほとんどないかもしれません。

つまり、

**予測に役立つ情報を特徴量として選ぶ**

ことが重要です。

---

# 💻 実習1：データを準備する

```python
import numpy as np

X = np.array([
    [1],
    [2],
    [3],
    [4],
    [5]
])

y = np.array([
    2,
    4,
    9,
    16,
    25
])
```

このデータは

```
x²
```

に近い形になっています。

普通の線形回帰では少し苦手なデータです。

---

# 💻 実習2：線形回帰を試す

```python
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X, y)

pred = model.predict(X)

print(pred)
```

予測はできますが、

完全にはフィットしません。

---

## なぜ？

線形回帰は

```
直線
```

しか表現できないからです。

---

# 💻 実習3：多項式特徴量

ここで

```python
from sklearn.preprocessing import PolynomialFeatures
```

を使います。

作成。

```python
poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)
```

変換。

```python
X_poly = poly.fit_transform(X)

print(X_poly)
```

結果。

```
[[1 1]
 [2 4]
 [3 9]
 [4 16]
 [5 25]]
```

つまり

```
x

↓

x

x²
```

という特徴量が追加されました。

---

# 💻 実習4：新しい特徴量で学習

```python
model = LinearRegression()

model.fit(
    X_poly,
    y
)

pred = model.predict(
    X_poly
)

print(pred)
```

先ほどより

かなり良く予測できるはずです。

---

## 中で起きていること

元々は

```
y = Wx+b
```

でした。

今は

```
x

x²
```

があるので

実際には

[
y
=

w_1x
+
w_2x^2
+
b
]

を学習しています。

線形回帰なのに

曲線を表現できる理由がこれです。

---

# 💻 実習5：Pipelineに組み込む

実務では

```python
from sklearn.pipeline import Pipeline
```

を使います。

```python
pipe = Pipeline([
    (
        "poly",
        PolynomialFeatures(
            degree=2,
            include_bias=False
        )
    ),
    (
        "model",
        LinearRegression()
    )
])
```

学習。

```python
pipe.fit(X, y)
```

予測。

```python
pred = pipe.predict(X)
```

これで

```
特徴量生成

↓

モデル学習
```

が一体化しました。

---

# 💻 実習6：特徴量が増える様子を見る

例えば

```python
X = np.array([
    [1,2]
])
```

に対して

```python
PolynomialFeatures(
    degree=2,
    include_bias=False
)
```

を使うと、

```python
print(
    poly.fit_transform(X)
)
```

概ね次のような特徴量になります。

```
x₁

x₂

x₁²

x₁x₂

x₂²
```

つまり

**特徴量同士の組み合わせ**

まで自動で作ってくれます。

---

# 📖 特徴量を増やせば良い？

答えは

**No**です。

例えば

```
100特徴量
```

なら、

degree=2

だけでも

かなり大量の特徴量になります。

すると

* 学習時間が長くなる
* 過学習しやすくなる
* メモリを多く使う

という問題が起こります。

これを

**特徴量爆発（Feature Explosion）**

と呼ぶことがあります。

---

# 💻 実習7：生成された特徴量名を見る

```python
poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)

poly.fit(X)
```

名前。

```python
print(
    poly.get_feature_names_out()
)
```

例えば

```
x0

x1

x0²

x0 x1

x1²
```

のような出力になります。

これにより、

どんな特徴量が追加されたか確認できます。

---

# ✍️ 演習

今日のデータです。

```python
X = np.array([
    [1],
    [2],
    [3],
    [4],
    [5],
    [6]
])

y = np.array([
    1,
    4,
    9,
    16,
    25,
    36
])
```

---

## 問1

線形回帰だけで学習してください。

---

## 問2

`PolynomialFeatures(degree=2)` を作ってください。

---

## 問3

`fit_transform()` を使って新しい特徴量を作ってください。

---

## 問4

多項式特徴量を使って線形回帰を学習してください。

---

## 問5

元のモデルと多項式特徴量を使ったモデルで、予測結果を比較してください。

---

## 問6（ボス戦👾）

次のPipelineを完成させてください。

```python
pipe = Pipeline([
    (
        "poly",
        PolynomialFeatures(
            degree=2,
            include_bias=False
        )
    ),
    (
        "model",
        LinearRegression()
    )
])
```

この `pipe` を使って学習・予測まで実行してください。

---

# 🌿 今日のまとめ

今日は、「AIに何を入力するか」がモデルの性能を左右することを学びました。

機械学習では、

```text
生データ
      ↓
特徴量エンジニアリング
      ↓
前処理
      ↓
モデル
      ↓
評価
```

という流れになります。

ここで重要なのは、

> **アルゴリズムだけでなく、特徴量も設計対象である**

という視点です。

これはデータサイエンスだけでなく、実務のAI開発でも非常に重要な考え方です。

---

# 🔜 第18回予告

次回は、機械学習モデルをより信頼して評価するための

> **Cross-validation（交差検証）**

を学びます。

テーマは、

* なぜ1回の train/test 分割では不十分なのか
* K-Fold Cross Validation
* `cross_val_score()` の使い方
* モデル比較の基本

です。

ここからは「たまたま良い結果だった」を避け、**再現性のあるモデル評価**へと一歩進みます。